In [8]:
# Import Libraries

import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras import regularizers
import gc
import golois

print ("Tensorflow version", tf.__version__)

Tensorflow version 2.15.0


In [9]:
# Configuration

planes = 31
moves = 361
N = 10000


input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype ('float32')

policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical (policy)

value = np.random.randint(2, size=(N,))
value = value.astype ('float32')

end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype ('float32')

groups = np.zeros((N, 19, 19, 1))
groups = groups.astype ('float32')

In [10]:
# Get Validation Data

print ("getValidation", flush = True)
golois.getValidation (input_data, policy, value, end)

getValidation


r.shape = (10000, 19, 19, 31)
nbExamples = 10000


In [11]:
from mobilenet_v2 import GoMobileNetv2
model = GoMobileNetv2((19,19,31), 361, 0.2)
#print(model.summary())

In [ ]:
from tensorflow.keras.utils import plot_model
from IPython.display import Image

# Création du modèle
model = get_model()
model.summary()

# Sauvegarde du schéma au format PNG
plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)
# Affichage dans le notebook
Image(filename='model.png')

In [ ]:
import time
import pandas as pd
from tensorflow.keras import optimizers, callbacks

# Epochs number
epochs = 250
batch = 32

def train_model(model):
  # Démarrer le chrono
  start_time = time.time()

  # Scheduler de learning rate (comme dans le papier)
  def get_learning_rate(epoch):
      if epoch < 100:
          return 0.0005
      elif epoch < 150:
          return 0.00005
      elif epoch < 200:
          return 0.000005
      else:
          return 0.0000005

  # Optimiseur Adam avec taux d’apprentissage initial
  # 💡 Optimiseur : SGD avec momentum
  optimizer = optimizers.SGD(learning_rate=get_learning_rate(0), momentum=0.9, nesterov=True)

  # Poids des pertes
  policy_weight = 1.0
  value_weight = 4.0

  # Pour stocker les métriques de chaque epoch
  all_history = []

  # Compilation du modèle
  model.compile(optimizer=optimizer,
                loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
                loss_weights={'policy': policy_weight, 'value': value_weight},
                metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

  # Entraînement avec scheduler
  for i in range(1, epochs + 1):

      # MAJ du learning rate manuellement
      lr = get_learning_rate(i)
      keras.backend.set_value(model.optimizer.learning_rate, lr)
      print('epoch ' + str(i)+ ', lr '+str(lr))

      # Mise à jour dynamique du batch
      golois.getBatch(input_data, policy, value, end, groups, i * N)

      history = model.fit(input_data,
                          {'policy': policy, 'value': value},
                          epochs=1,
                          batch_size=batch)

      # Stocker l’historique
      metrics = {key: val[0] for key, val in history.history.items()}
      metrics['epoch'] = i
      all_history.append(metrics)

      if i % 5 == 0:
          gc.collect()

      #if i % 20 == 0:
      golois.getValidation(input_data, policy, value, end)
      val = model.evaluate(input_data,
                              [policy, value], verbose=0, batch_size=batch)

  total_time = time.time() - start_time
  return val, pd.DataFrame(all_history), total_time

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def plot_result(history_dfs, labels, epochs=None):
    assert len(history_dfs) == len(labels)

    # Titre
    info = []
    title = f"Epochs: {epochs}"

    # Grille personnalisée : 2 lignes (3 en haut, 2 en bas)
    fig = plt.figure(figsize=(18, 8))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 3)

    # --- Ligne 1 : 3 plots ---
    ax1 = fig.add_subplot(gs[0, 0])
    for df, label in zip(history_dfs, labels):
        ax1.plot(df['epoch'], df['loss'], label=f'{label} Total Loss')
    ax1.set_title('Total Loss par Epoch')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Total Loss')
    ax1.legend()

    ax2 = fig.add_subplot(gs[0, 1])
    for df, label in zip(history_dfs, labels):
        if 'policy_loss' in df.columns:
            ax2.plot(df['epoch'], df['policy_loss'], label=f'{label} Policy Loss')
    ax2.set_title('Policy Loss par Epoch')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Policy Loss')
    ax2.legend()

    ax3 = fig.add_subplot(gs[0, 2])
    for df, label in zip(history_dfs, labels):
        if 'value_loss' in df.columns:
            ax3.plot(df['epoch'], df['value_loss'], label=f'{label} Value Loss')
    ax3.set_title('Value Loss par Epoch')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Value Loss')
    ax3.legend()

    # --- Ligne 2 : 2 plots ---
    ax4 = fig.add_subplot(gs[1, 0])
    for df, label in zip(history_dfs, labels):
        if 'policy_categorical_accuracy' in df.columns:
            ax4.plot(df['epoch'], df['policy_categorical_accuracy'], label=label)
    ax4.set_title('Policy Accuracy par Epoch')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Categorical Accuracy')
    ax4.legend()

    ax5 = fig.add_subplot(gs[1, 1])
    for df, label in zip(history_dfs, labels):
        if 'value_mse' in df.columns:
            ax5.plot(df['epoch'], df['value_mse'], label=label)
    ax5.set_title('Value MSE par Epoch')
    ax5.set_xlabel('Epoch')
    ax5.set_ylabel('MSE')
    ax5.legend()

    # Libérer la dernière case vide de la 2e ligne
    fig.delaxes(fig.add_subplot(gs[1, 2]))  # case vide propre

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

def print_validation_results(model_results, epoch=epochs):
    """
    Affiche les résultats de validation pour une liste de modèles.

    Paramètres :
    - model_results : liste de tuples (model, val, label)
    - epoch : (optionnel) numéro d'epoch à afficher dans le titre
    """
    for model, val, label, time in model_results:
        metrics = dict(zip(model.metrics_names, val))
        title = f"📊 Validation Results for {label}"
        if epoch is not None:
            title += f" — Epoch {epoch}"
        print(f"\n{title}:")
        for name, value in metrics.items():
            print(f"  - {name:<30}: {value:.4f}")
        print(f"  - Time: {time:.4f}")

In [ ]:
from google.colab import drive

val, all_history, total_time = train_model(model)

# Affichage des résultats
results = [
    (model, val, "Nesterov", total_time)#,
]
print_validation_results(results)

# Affichage des courbes comparatives
plot_result(
    history_dfs=[all_history],
    labels=["Nesterov"],
    epochs=epochs
)

drive.mount('/content/drive')
model.save('/content/drive/MyDrive/mchettih.h5')